In [ ]:
!pip install -q gradio transformers torch keybert sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.6 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import torch
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from keybert import KeyBERT


# ============================================
# LOAD MODELS
# ============================================

print("Loading summarization model...")

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Loading keyword model...")

kw_model = KeyBERT("all-MiniLM-L6-v2")

print("Models loaded successfully!")


# ============================================
# TEXT ANALYSIS FUNCTION
# ============================================

def analyze_text(text):

    if not text or not text.strip():
        return (
            "Please enter some text.",
            "No keywords found.",
            0,
            0
        )

    text = text.strip()

    # ----------------------------------------
    # Word Count
    # ----------------------------------------

    words = text.split()
    word_count = len(words)

    # ----------------------------------------
    # Sentence Count
    # ----------------------------------------

    sentences = re.split(r'[.!?]+', text)
    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    sentence_count = len(sentences)

    # ----------------------------------------
    # TEXT SUMMARIZATION
    # ----------------------------------------

    if word_count < 30:

        summary = text

    else:

        # Tokenize input
        inputs = tokenizer(
            text,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )

        # Generate summary
        with torch.no_grad():

            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=150,
                min_length=40,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

    # ----------------------------------------
    # KEYWORD EXTRACTION
    # ----------------------------------------

    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=8
    )

    keyword_list = ", ".join(
        [keyword for keyword, score in keywords]
    )

    if not keyword_list:
        keyword_list = "No keywords found."

    # ----------------------------------------
    # RETURN RESULTS
    # ----------------------------------------

    return (
        summary,
        keyword_list,
        word_count,
        sentence_count
    )


# ============================================
# GRADIO USER INTERFACE
# ============================================

with gr.Blocks(
    title="Text Summarizer & Keyword Extractor"
) as app:

    gr.Markdown(
        """
        # 📝 Text Summarizer & Keyword Extractor

        Enter a paragraph or article below.

        The application will:
        - Generate a short summary
        - Extract important keywords
        - Count words
        - Count sentences
        """
    )

    # Input
    text_input = gr.Textbox(
        label="📄 Enter Your Text",
        placeholder="Paste your article or paragraph here...",
        lines=12
    )

    # Button
    analyze_button = gr.Button(
        "🔍 Analyze Text",
        variant="primary"
    )

    gr.Markdown("## 📊 Analysis Results")

    # Summary
    summary_output = gr.Textbox(
        label="📌 Generated Summary",
        lines=7
    )

    # Keywords
    keywords_output = gr.Textbox(
        label="🔑 Important Keywords",
        lines=2
    )

    # Statistics
    with gr.Row():

        word_output = gr.Number(
            label="📝 Word Count"
        )

        sentence_output = gr.Number(
            label="📌 Sentence Count"
        )

    # Connect button
    analyze_button.click(
        fn=analyze_text,
        inputs=text_input,
        outputs=[
            summary_output,
            keywords_output,
            word_output,
            sentence_output
        ]
    )


# ============================================
# START APPLICATION
# ============================================

app.launch(share=True)

Loading summarization model...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b85281e47154016a3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
